# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipanshurdev/ML-Assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton maps our chosen lane onto the machine learning loop: task formulation, target definition, success metrics, unit of analysis, and the rationale for machine learning over static heuristic rules.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Chosen Lane: CTR / Engagement Opportunity Scoring & Ranking
We address the operational content triage problem: identifying high-visibility web pages that fail to convert search impressions into organic clicks.

### ML Task Type: Binary Classification + Probability-Based Ranking / Scoring
- **Classification Component:** Predict whether a page ranking on Page 1 of Google Search suffers from an acute click-through rate (CTR) deficit ($y \in \{0, 1\}$).
- **Ranking / Scoring Component:** We leverage the model's calibrated posterior probability estimates ($P(y=1 \mid \mathbf{x})$) to score, sort, and prioritize candidate URLs into an actionable weekly editorial triage queue.
- **Why this formulation fits:**
  1. Content marketing teams have strictly finite editorial capacity; they cannot review thousands of URLs and require an ordered triage queue where the most severe, high-leverage underperformers appear at the top.
  2. Classification provides an interpretable performance boundary, while continuous probability scoring enables flexible capacity batching (e.g., pulling the top 20 or top 50 pages each review cycle).

In [1]:
# Task formulation specification
task_spec = {
    "Lane": "CTR / Engagement Opportunity Scoring",
    "ML Task Type": "Binary Classification & Probability Ranking",
    "Decision Supported": "Prioritizing content pages for Title & Meta Description rewrites",
    "Who Acts": "SEO Strategists and Content Editors",
    "Intervention": "Non-destructive SERP snippet optimization (~15 mins per URL)",
    "Cost of False Positive": "Low (~5 mins editor inspection, confirms snippet is fine)",
    "Cost of False Negative": "High (continuous traffic and revenue loss on high-intent queries)"
}

for k, v in task_spec.items():
    print(f"{k:25}: {v}")

Lane                     : CTR / Engagement Opportunity Scoring
ML Task Type             : Binary Classification & Probability Ranking
Decision Supported       : Prioritizing content pages for Title & Meta Description rewrites
Who Acts                 : SEO Strategists and Content Editors
Intervention             : Non-destructive SERP snippet optimization (~15 mins per URL)
Cost of False Positive   : Low (~5 mins editor inspection, confirms snippet is fine)
Cost of False Negative   : High (continuous traffic and revenue loss on high-intent queries)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target Definition:
A binary indicator ($y \in \{0, 1\}$) denoting whether a content URL ranking on Google Page 1 ($\text{avg\_position} \le 10$) with substantial search demand ($\text{impressions} \ge 1,000$) exhibits an acute CTR anomaly below 1.5%:
$$y = \mathbb{I}(\text{ctr} < 1.5\%)$$

### Origin of the Label: Observed Performance Outcome
- **Observed Telemetry:** The label is computed from actual, measured searcher interactions recorded by Google Search Console over a trailing measurement window (total clicks divided by total impressions).
- **Why it is an observed outcome:** It reflects empirical user behavior in search results (users viewed the snippet in SERP but selected competing listings), rather than an arbitrary subjective tag or human survey score.
- **Feature Leakage Safeguard:** Because $\text{ctr} = \frac{\text{clicks}}{\text{impressions}}$, direct click counts (`clicks_90d`, `gsc_clicks`) directly reveal the ground-truth label. Therefore, raw clicks are **strictly excluded** from the feature vector during modeling.

In [2]:
# Demonstrate target definition logic on starter slice
import pandas as pd
import numpy as np
import os

# Robust path resolution across environments (Colab, root, notebooks dir)
possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    'https://raw.githubusercontent.com/dipanshurdev/ML-Assignments/main/data/raw/content_refresh_anonymized.csv'
]

data_path = None
for p in possible_paths:
    if p.startswith('http') or os.path.exists(p):
        data_path = p
        break

df = pd.read_csv(data_path)

# Filter to qualified population (Page 1 with statistical demand)
# Note: In the starter dataset, rate columns like ctr are percentage values (e.g., 0.76 means 0.76%)
qualified_df = df[(df['avg_position'] > 0) & (df['avg_position'] <= 10) & (df['impressions_90d'] >= 1000)].copy()

# Construct observed binary target: 1 = CTR Anomaly (<1.5%), 0 = Healthy CTR
qualified_df['target_anomaly'] = (qualified_df['ctr'] < 1.5).astype(int)

print("Target Label Distribution:")
print(qualified_df['target_anomaly'].value_counts(normalize=True).rename(lambda x: f"Class {x} ({'Anomaly' if x==1 else 'Normal'})").apply(lambda x: f"{x:.2%}"))
print(f"\nTotal Candidate URLs: {len(qualified_df):,}")

Target Label Distribution:
target_anomaly
Class 1 (Anomaly)    98.51%
Class 0 (Normal)      1.49%
Name: proportion, dtype: object

Total Candidate URLs: 6,591


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Primary Success Metric: Precision@K (Specifically Precision@20 and Precision@50)
- **Why Precision@K?**
  Editorial review capacity is strictly capacity-constrained: content teams audit and rewrite a fixed batch size ($K=20$ or $K=50$ URLs) per review cycle. 
  Precision@K measures the proportion of recommended URLs in the top $K$ queue that are true, high-impact CTR anomalies:
  $$\text{Precision@}K = \frac{\sum_{i=1}^K y_{(i)}}{K}$$
- **What number means 'good'?**
  - **Empirical Base Rate:** Random selection from the Page 1 candidate pool defines the base rate benchmark.
  - **Target Standard:** A model achieving **$\ge 95\%$ Precision@20** (matching or outperforming a domain heuristic baseline) on held-out client domains provides trustworthy operational value without wasting human editorial hours on false alarms.
- **Secondary Metrics:**
  - **ROC-AUC & PR-AUC:** To quantify ranking discrimination across the entire candidate spectrum.
  - **Lift over Random:** Demonstrating concentration of true anomalies in top percentiles.

In [3]:
# Precision@K metric calculation definition
def precision_at_k(y_true, y_scores, k=20):
    """Calculates Precision@K given ground truth and predicted scores."""
    top_k_indices = np.argsort(y_scores)[::-1][:k]
    return np.mean(np.array(y_true)[top_k_indices])

# Benchmark base rate
base_rate = qualified_df['target_anomaly'].mean()
print(f"Empirical Base Rate on Qualified Population: {base_rate:.2%}")
print(f"Target Performance Bar: Precision@20 >= 95.0% on held-out client splits")

Empirical Base Rate on Qualified Population: 98.51%
Target Performance Bar: Precision@20 >= 95.0% on held-out client splits


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis:
**One row = One unique pseudonymized content item / URL (`content_id`)** belonging to a client domain (`client_id`), aggregated across a 90-day search and on-site engagement observation window.

In the production warehouse release, this grain corresponds to `fact_content_daily_performance` aggregated by `client_hash_id` and `content_hash_id`.

In [4]:
# Display the unit of analysis as a clean dataframe slice
preview_columns = [
    'content_id', 'client_id', 'avg_position', 
    'impressions_90d', 'sessions_90d', 'ctr', 'target_anomaly'
]

sample_slice = qualified_df[preview_columns].head(10)
print(f"Dataframe Slice Shape: {qualified_df.shape[0]:,} rows x {qualified_df.shape[1]} columns")
print(f"Unique Content Items (Grain): {qualified_df['content_id'].nunique():,}")
print(f"Unique Clients: {qualified_df['client_id'].nunique()}")
print("\nUnit of Analysis Preview (1 row = 1 content item):")
display(sample_slice)

Dataframe Slice Shape: 6,591 rows x 45 columns
Unique Content Items (Grain): 6,591
Unique Clients: 28

Unit of Analysis Preview (1 row = 1 content item):


,content_id,client_id,avg_position,impressions_90d,sessions_90d,ctr,target_anomaly
3,content_331d6c4de07b,client_19581e27de,6.2,11751,78,0.49,1
5,content_d4084a4bc775,client_f369cb89fc,8.5,3970,5,0.03,1
9,content_c27558df2b0c,client_19581e27de,4.9,1240,3,0.16,1
10,content_d8ee6cc6d642,client_19581e27de,2.2,20919,326,1.55,0
12,content_42fb2cad9ecf,client_6208ef0f77,5.6,7228,233,1.76,0
16,content_78bd1d4a1d4d,client_6208ef0f77,8.9,13848,475,0.15,1
17,content_761a44afda12,client_19581e27de,7.3,9449,118,0.07,1
20,content_0d748c484ab1,client_f369cb89fc,5.6,2538,13,0.55,1
22,content_3fb46bec4413,client_4e07408562,3.9,2311,5,0.13,1
24,content_0e23e310d404,client_19581e27de,4.1,29541,108,0.27,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A naive approach might suggest using a simple deterministic rule:
`IF impressions >= 1000 AND avg_position <= 10 AND ctr < 1.5% THEN rewrite`

Why machine learning fundamentally outperforms this fixed rule:
1. **Steep Non-Linear Rank-to-CTR Relationship:**
   Search CTR follows a steep power-law decay curve across Page 1 rankings:
   - Rank 1.0 expects ~25%–30% CTR.
   - Rank 3.0 expects ~8%–12% CTR.
   - Rank 9.0 expects ~1.5%–2.0% CTR.
   A static 1.5% threshold fails catastrophically for top-ranking pages: an article ranking at **Position 1.5 with 2.2% CTR** is a massive anomaly bleeding thousands of clicks, yet a static `< 1.5%` rule completely ignores it! ML models learn rank-conditioned expectations naturally.
2. **Multi-Modal Signal Disconnect (Search vs. On-Site Sessions):**
   A true SERP snippet failure shows high search impressions but disproportionately low on-site sessions (`ga4_sessions`). Combining search exposure, rank, and on-site behavioral signals involves tangled interaction effects that static if-statements cannot calibrate without brittle over-engineering.
3. **Continuous Probability Calibration vs. Unordered Set:**
   A fixed rule returns an unordered binary bucket of hundreds of matching URLs with zero priority ranking. A probabilistic model (e.g., Random Forest) outputs continuous probabilities ($P \in [0, 1]$), allowing teams to rank candidates by severity and confidence.

In [5]:
# Illustrate non-linear CTR expectation across rank
rank_summary = qualified_df.groupby(
    pd.cut(qualified_df['avg_position'], bins=[0, 2, 4, 7, 10], labels=['Top 2 (1-2)', 'Positions 3-4', 'Mid Page (5-7)', 'Bottom (8-10)']),
    observed=False
).agg(
    url_count=('content_id', 'count'),
    mean_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    mean_impressions=('impressions_90d', 'mean')
).reset_index()

print("Empirical CTR Distribution Across SERP Ranks (Demonstrating Non-Linearity):")
print(rank_summary.to_string(index=False))

Empirical CTR Distribution Across SERP Ranks (Demonstrating Non-Linearity):
  avg_position  url_count  mean_ctr  median_ctr  mean_impressions
   Top 2 (1-2)        108  0.295278        0.14      15861.666667
 Positions 3-4        909  0.429395        0.33      19307.896590
Mid Page (5-7)       3152  0.353493        0.26      17169.260787
 Bottom (8-10)       2422  0.305376        0.20       9214.221718


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.